In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet


In [2]:
!pip install -qU rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 51.5 MB/s eta 0:00:00


In [3]:
import os, math, random, time, warnings, re
from collections import OrderedDict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, inchi, rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

import rdkit
print("RDKit:", rdkit.__version__, "| Torch:", torch.__version__)

RDKit: 2026.03.6 | Torch: 2.11.0+cu128


In [4]:
class CFG:
    DEMO = False                      # True = toy synthetic data. False = real data.
    SEED = 420

    # ---- real data paths (used when DEMO = False) ----
    TRAIN_PATH = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet"   # or .parquet / .mgf
    TEST_PATH  = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet"
    EXTRA_LIB_PATHS = []              # files with a column/line of SMILES: [".csv", ".parquet", ".smi", ".txt"]
    EXTRA_LIB_SMILES_COL = "smiles"
    SUBMISSION_PATH = "submission.csv"

    # ---- spectrum preprocessing ----
    MAX_PEAKS = 64
    MIN_REL_INT = 0.005               # drop peaks below 0.5% of base peak
    MAX_MZ = 1500.0

    # ---- fingerprint ----
    NBITS = 2048
    FP_RADIUS = 2

    # ---- model ----
    D_MODEL = 256
    N_LAYERS = 6
    N_HEADS = 8
    FF_MULT = 4
    DROPOUT = 0.1
    EMB_DIM = 128                     # contrastive embedding size

    # ---- training ----
    EPOCHS = 7
    BATCH_SIZE = 512
    LR = 1e-3
    WEIGHT_DECAY = 1e-2
    WARMUP_FRAC = 0.05
    CONTRASTIVE_W = 0.5
    PEAK_DROPOUT = 0.10               # augmentation
    MZ_JITTER = 0.002                 # Da, augmentation
    VAL_FRAC = 0.10
    EVAL_EVERY = 5
    EVAL_MAX_N = 300                  # max val spectra used for MRR checks during training
    GRAD_CLIP = 1.0
    NUM_WORKERS = 8

    # ---- retrieval ----
    PPM_TIERS = [10, 30, 100, 500]    # widen the mass window until 25 distinct candidates exist
    TOPK = 25
    ALPHA = 0.5                       # weight of contrastive cosine vs fingerprint log-likelihood
    # In validation, should training molecules also be in the candidate library?
    # False mimics "novel molecule" test conditions.
    VAL_LIB_INCLUDES_TRAIN = False

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

seed_everything(CFG.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print("WARNING: no GPU found. Training will be slow. Enable a GPU runtime.")

# bf16 if the GPU supports it, otherwise fp16 with a grad scaler
USE_AMP = device.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
print("AMP:", USE_AMP, AMP_DTYPE)

Device: cuda
GPU: Tesla T4 | 15.6 GB
AMP: True torch.bfloat16


In [5]:
_taut = rdMolStandardize.TautomerEnumerator()
_IK_CACHE = {}

def ik14(smiles):
    """InChIKey14 after RDKit tautomer canonicalization. None if the SMILES is invalid."""
    if smiles in _IK_CACHE:
        return _IK_CACHE[smiles]
    out = None
    try:
        m = Chem.MolFromSmiles(smiles)
        if m is not None:
            m = _taut.Canonicalize(m)
            out = inchi.MolToInchiKey(m)[:14]
    except Exception:
        out = None
    _IK_CACHE[smiles] = out
    return out

def mrr_at_k(pred_lists, true_smiles, k=25):
    """pred_lists: list of ranked SMILES lists. true_smiles: list of answers."""
    rr, top1, top5, hit = [], 0, 0, 0
    for preds, truth in zip(pred_lists, true_smiles):
        t = ik14(truth)
        r = 0.0
        for rank, p in enumerate(preds[:k], 1):
            if t is not None and ik14(p) == t:
                r = 1.0 / rank
                top1 += rank == 1
                top5 += rank <= 5
                hit += 1
                break
        rr.append(r)
    n = max(len(rr), 1)
    return {"MRR@25": float(np.mean(rr)) if rr else 0.0,
            "top1": top1 / n, "top5": top5 / n, "recall@25": hit / n}

# Sanity check on the glucose example from the competition page
_glc_a = "OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O"
_glc_b = "C(C1C(C(C(C(O1)O)O)O)O)O"
print("glucose InChIKey14:", ik14(_glc_a), ik14(_glc_b), "| match:", ik14(_glc_a) == ik14(_glc_b))

# ---- adducts: mass shift from neutral M to the ion (ion m/z = M + delta) ----
PROTON = 1.007276
ADDUCT_DELTA = {
    "[M+H]+": PROTON,
    "[M+NH4]+": 18.033823,
    "[M+Na]+": 22.989218,
    "[M+K]+": 38.963158,
    "[M+H-H2O]+": PROTON - 18.010565,
    "[M-H]-": -PROTON,
    "[M+Cl]-": 34.969402,
    "[M+FA-H]-": 44.998201,
}
ADDUCT_LIST = list(ADDUCT_DELTA.keys())
ADDUCT_ID = {a: i for i, a in enumerate(ADDUCT_LIST)}

def _adduct_core(a):
    """'[M+H]+' -> 'M+H', 'M+H' -> 'M+H', '[M-H]-' -> 'M-H'."""
    a = re.sub(r"[\[\]\s]", "", str(a))
    return re.sub(r"[+-]$", "", a) if a.count("+") + a.count("-") >= 2 else a

def norm_adduct(a, mode=None):
    """Map free-text adducts (e.g. 'M+H', '[M+H]+') to a key of ADDUCT_DELTA."""
    if isinstance(a, str):
        a = a.strip()
        if a in ADDUCT_DELTA:
            return a
        core = _adduct_core(a)
        for cand in ADDUCT_LIST:
            if _adduct_core(cand) == core:
                return cand
    if mode is not None and str(mode).lower().startswith("neg"):
        return "[M-H]-"
    return "[M+H]+"

_mfg = rdFingerprintGenerator.GetMorganGenerator(radius=CFG.FP_RADIUS, fpSize=CFG.NBITS)

def smiles_fp(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return None
    return _mfg.GetFingerprintAsNumPy(m).astype(np.uint8)

def flat_smiles(smi):
    """Plain canonical SMILES without stereo. Cheap de-dup key for library building."""
    m = Chem.MolFromSmiles(smi)
    return None if m is None else Chem.MolToSmiles(m, isomericSmiles=False)

def scaffold_of(smi):
    try:
        m = Chem.MolFromSmiles(smi)
        s = MurckoScaffold.GetScaffoldForMol(m)
        g = MurckoScaffold.MakeScaffoldGeneric(s)
        return Chem.MolToSmiles(g)
    except Exception:
        return ""

glucose InChIKey14: WQZGKKKJIJFFOK WQZGKKKJIJFFOK | match: True


In [6]:
import gc, psutil
from functools import lru_cache

def mem(tag=""):
    s = f"[{tag}] RAM used {psutil.Process().memory_info().rss/1e9:.2f} GB | free {psutil.virtual_memory().available/1e9:.2f} GB"
    if device.type == "cuda":
        s += f" | VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB"
    print(s)

def free():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

# Cap the tautomer cache (replaces the old dict-based ik14 and _IK_CACHE)
@lru_cache(maxsize=2_000_000)
def ik14(smiles):
    try:
        m = Chem.MolFromSmiles(smiles)
        if m is not None:
            m = _taut.Canonicalize(m)
            return inchi.MolToInchiKey(m)[:14]
    except Exception:
        pass
    return None

# Streaming parallel map: results arrive in order and can be written out immediately
def parallel_imap(fn, tasks, desc="", n_jobs=None):
    n_jobs = n_jobs or N_JOBS
    ctx = None
    if n_jobs > 1 and len(tasks) > 1:
        try: ctx = mp.get_context("fork")
        except ValueError: ctx = None
    if ctx is None:
        for t in tqdm(tasks, desc=desc, leave=False): yield fn(t)
        return
    with ctx.Pool(min(n_jobs, len(tasks))) as pool:
        for r in tqdm(pool.imap(fn, tasks), total=len(tasks), desc=desc, leave=False):
            yield r

# Unpack np.packbits rows on the GPU: (B, 512) uint8 -> (B, 4096) 0/1
_SHIFTS = {}
def unpack_gpu(packed):
    dev = packed.device
    if dev not in _SHIFTS:
        _SHIFTS[dev] = torch.arange(7, -1, -1, device=dev, dtype=torch.int16)
    bits = (packed.unsqueeze(-1).to(torch.int16) >> _SHIFTS[dev]) & 1
    return bits.flatten(1)

In [7]:
COLS = dict(
    id="id",  # Not present in this file; standardize_df will generate IDs.
    smiles="normalized_smiles",
    precursor_mz="precursor_mz",
    adduct="adduct",  # Not present; standardize_df infers it from ionization_mode.
    ion_mode="ionization_mode",
    collision_energy="collision_energy",
    mz="ms2_mzs",
    intensity="ms2_normalized_intensities",
)
def _to_arr(x):
    if isinstance(x, str):
        x = [float(v) for v in re.split(r"[,\s;\[\]]+", x.strip()) if v]
    return np.asarray(x, dtype=np.float32)

def read_mgf(path):
    rows, cur, peaks = [], None, []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.upper() == "BEGIN IONS":
                cur, peaks = {}, []
            elif line.upper() == "END IONS":
                if peaks:
                    arr = np.asarray(peaks, dtype=np.float32)
                    cur["mz"], cur["intensity"] = arr[:, 0], arr[:, 1]
                rows.append(cur); cur = None
            elif cur is not None:
                if "=" in line and not line[0].isdigit():
                    k, v = line.split("=", 1)
                    cur[k.strip().lower()] = v.strip()
                else:
                    p = line.split()
                    peaks.append((float(p[0]), float(p[1])))
    df = pd.DataFrame(rows)
    ren = {"pepmass": "precursor_mz", "ionmode": "ion_mode", "collisionenergy": "collision_energy"}
    df = df.rename(columns={k: v for k, v in ren.items() if k in df.columns})
    for k in ("feature_id", "title"):          # first one found becomes the id
        if k in df.columns:
            df = df.rename(columns={k: "id"}); break
    if "precursor_mz" in df.columns:
        df["precursor_mz"] = df["precursor_mz"].astype(str).str.split().str[0].astype(float)
    return df

def read_table(path):
    p = path.lower()
    if p.endswith(".mgf"):
        return read_mgf(path)
    if p.endswith(".parquet"):
        return pd.read_parquet(path)
    if p.endswith(".json") or p.endswith(".jsonl"):
        return pd.read_json(path, lines=p.endswith(".jsonl"))
    return pd.read_csv(path)

def standardize_df(df):
    """Rename to canonical columns and fill defaults."""
    inv = {v: k for k, v in COLS.items() if v in df.columns}
    df = df.rename(columns=inv).copy()
    if "id" not in df.columns:
        df["id"] = np.arange(len(df))
    if "ion_mode" not in df.columns:
        df["ion_mode"] = "positive"
    if "collision_energy" not in df.columns:
        df["collision_energy"] = 0.0
    if "adduct" not in df.columns:
        df["adduct"] = [("[M-H]-" if str(m).lower().startswith("neg") else "[M+H]+") for m in df["ionization_mode"]]
    df["adduct"] = [norm_adduct(a, m) for a, m in zip(df["adduct"], df["ion_mode"])]
    df["collision_energy"] = pd.to_numeric(df["collision_energy"], errors="coerce").fillna(0.0)
    df["mz"] = [_to_arr(x) for x in df["mz"]]
    df["intensity"] = [_to_arr(x) for x in df["intensity"]]
    return df.reset_index(drop=True)

In [8]:
t0 = time.time()
if CFG.DEMO:
    train_df, test_df, extra_smiles = make_demo()
    test_truth = test_df["smiles"].tolist()      # demo only: lets us score the "test" set locally
else:
    train_df = standardize_df(read_table(CFG.TRAIN_PATH))
    test_df = read_table(CFG.TEST_PATH)#standardize_df(read_table(CFG.TEST_PATH))
    test_truth = None
    extra_smiles = []
    for p in CFG.EXTRA_LIB_PATHS:
        if p.endswith((".smi", ".txt")):
            extra_smiles += [l.split()[0] for l in open(p) if l.strip()]
        else:
            extra_smiles += read_table(p)[CFG.EXTRA_LIB_SMILES_COL].dropna().tolist()

if CFG.DEMO:
    train_df = standardize_df(train_df); test_df = standardize_df(test_df)

print(f"train {len(train_df)} | test {len(test_df)} | extra library SMILES {len(extra_smiles)} | {time.time()-t0:.1f}s")
print(train_df[["id", "precursor_mz", "adduct", "collision_energy"]].head())
print("adducts:", train_df["adduct"].value_counts().to_dict())

train 2539608 | test 1213 | extra library SMILES 0 | 65.8s
   id  precursor_mz   adduct  collision_energy
0   0      330.1237   [M+H]+               0.0
1   1      432.1320   [M+K]+               0.0
2   2      422.1738  [M+Na]+               0.0
3   3      285.0789   [M+H]+               0.0
4   4      746.0984   [M+H]+               0.0
adducts: {'[M+H]+': 1644568, '[M-H]-': 617030, '[M+Na]+': 178092, '[M+NH4]+': 53452, '[M+Cl]-': 27511, '[M+K]+': 18955}


In [9]:
import multiprocessing as mp

N_JOBS = max(1, (os.cpu_count() or 1) - 1)

def parallel_map(fn, tasks, desc="", n_jobs=N_JOBS):
    """Ordered parallel map. Uses fork (Linux/Colab/Kaggle). Falls back to serial elsewhere."""
    ctx = None
    if n_jobs > 1 and len(tasks) > 1:
        try:
            ctx = mp.get_context("fork")
        except ValueError:
            ctx = None
    if ctx is None:
        return [fn(t) for t in tqdm(tasks, desc=desc, leave=False)]
    with ctx.Pool(min(n_jobs, len(tasks))) as pool:
        return list(tqdm(pool.imap(fn, tasks), total=len(tasks), desc=desc, leave=False))

# ---------------- spectra -> tensors ----------------
_SPEC_G = {}   # set before the pool forks, so workers read it without pickling

def _spec_rows(rng_):
    a, b = rng_
    mzs, ints, pmzs = _SPEC_G["mz"], _SPEC_G["int"], _SPEC_G["prec"]
    P, n = CFG.MAX_PEAKS, b - a
    mz = np.zeros((n, P), np.float32); it = np.zeros((n, P), np.float32); pad = np.ones((n, P), bool)
    for r in range(n):
        m, s, pmz = mzs[a + r], ints[a + r], pmzs[a + r]
        if len(m) == 0:
            pad[r, 0] = False; continue
        s = s / (s.max() + 1e-9)
        keep = (s >= CFG.MIN_REL_INT) & (m <= min(pmz + 5.0, CFG.MAX_MZ))
        m, s = m[keep], s[keep]
        if len(m) > P:
            top = np.argsort(-s)[:P]; m, s = m[top], s[top]
        o = np.argsort(m); m, s = m[o], s[o]
        k = len(m)
        mz[r, :k], it[r, :k], pad[r, :k] = m, s, False      # pad=True means padding
        if k == 0:
            pad[r, 0] = False
    return mz, it, pad

def spectra_to_tensors(df, n_jobs=N_JOBS):
    N = len(df)
    _SPEC_G.update(mz=list(df["mz"]), int=list(df["intensity"]),
                   prec=df["precursor_mz"].values.astype(np.float64))
    step = max(256, math.ceil(N / (n_jobs * 8)))
    ranges = [(i, min(i + step, N)) for i in range(0, N, step)]
    parts = parallel_map(_spec_rows, ranges, "spectra", n_jobs)
    mz = np.concatenate([p[0] for p in parts]); it = np.concatenate([p[1] for p in parts])
    pad = np.concatenate([p[2] for p in parts])
    _SPEC_G.clear()
    return dict(
        mz=torch.from_numpy(mz), inten=torch.from_numpy(it), pad=torch.from_numpy(pad),
        prec=torch.tensor(df["precursor_mz"].values, dtype=torch.float32),
        adduct=torch.tensor([ADDUCT_ID[a] for a in df["adduct"]], dtype=torch.long),
        ce=torch.tensor(df["collision_energy"].values, dtype=torch.float32),
    )

# ---------------- SMILES -> fingerprints ----------------
_W_MFG = None   # one generator per worker process

def _fp_chunk(smiles_chunk):
    global _W_MFG
    if _W_MFG is None:
        _W_MFG = rdFingerprintGenerator.GetMorganGenerator(radius=CFG.FP_RADIUS, fpSize=CFG.NBITS)
    out = np.zeros((len(smiles_chunk), CFG.NBITS), np.uint8)
    ok = np.ones(len(smiles_chunk), bool)
    for i, s in enumerate(smiles_chunk):
        m = Chem.MolFromSmiles(s)
        if m is None:
            ok[i] = False
        else:
            out[i] = _W_MFG.GetFingerprintAsNumPy(m)
    return np.packbits(out, axis=1), ok          # packed 8x to keep inter-process traffic small

def _scaffold_chunk(smiles_chunk):
    return [scaffold_of(s) for s in smiles_chunk]

def scaffold_split(smiles, val_frac, seed=0, n_jobs=N_JOBS):
    # 1) scaffolds for unique SMILES only, in parallel
    uniq = list(dict.fromkeys(smiles))
    step = max(200, math.ceil(len(uniq) / (n_jobs * 8)))
    chunks = [uniq[i:i + step] for i in range(0, len(uniq), step)]
    res = parallel_map(_scaffold_chunk, chunks, "scaffolds", n_jobs)
    scaf_of = dict(zip(uniq, [s for r in res for s in r]))

    # 2) group row indices by scaffold (cheap dict lookups)
    groups = {}
    for i, s in enumerate(smiles):
        groups.setdefault(scaf_of[s], []).append(i)

    # 3) same selection logic as before
    keys = list(groups); random.Random(seed).shuffle(keys)
    n_val, val_idx = int(len(smiles) * val_frac), []
    for k in keys:
        if len(val_idx) >= n_val: break
        val_idx += groups[k]

    is_val = np.zeros(len(smiles), bool); is_val[val_idx] = True
    return np.flatnonzero(~is_val), np.flatnonzero(is_val)
def fp_matrix_packed(smiles_list, n_jobs=N_JOBS):
    """One packed row per UNIQUE molecule, plus an index from each spectrum to its molecule."""
    uniq = list(dict.fromkeys(smiles_list))
    step = max(200, math.ceil(len(uniq) / (n_jobs * 8)))
    tasks = [uniq[i:i + step] for i in range(0, len(uniq), step)]
    packed, oks = [], []
    for p, o in parallel_imap(_fp_chunk, tasks, "fingerprints", n_jobs):
        packed.append(p); oks.append(o)
    packed_u, ok_u = np.concatenate(packed), np.concatenate(oks)
    pos = {s: i for i, s in enumerate(uniq)}
    mol_idx = np.fromiter((pos[s] for s in smiles_list), dtype=np.int64, count=len(smiles_list))
    return packed_u, mol_idx, ok_u[mol_idx]

packed_u, mol_idx, ok = fp_matrix_packed(train_df["smiles"].tolist())
train_df = train_df[ok].reset_index(drop=True); mol_idx = mol_idx[ok]
print("usable training spectra:", len(train_df), "| unique molecules:", len(packed_u))

tr_idx, va_idx = scaffold_split(train_df["smiles"].tolist(), CFG.VAL_FRAC)
print("train:", len(tr_idx), "| val:", len(va_idx))
tr_df, va_df = train_df.iloc[tr_idx].reset_index(drop=True), train_df.iloc[va_idx].reset_index(drop=True)
tr_mol, va_mol = mol_idx[tr_idx], mol_idx[va_idx]

test_df = standardize_df(test_df)
tr_T, va_T, te_T = spectra_to_tensors(tr_df), spectra_to_tensors(va_df), spectra_to_tensors(test_df)

# The peak arrays now live in the tensors. Drop them from every DataFrame.
for d in (train_df, tr_df, va_df, test_df):
    d.drop(columns=["mz", "intensity"], inplace=True, errors="ignore")
free(); mem("after tensors")

fingerprints:   0%|          | 0/24 [00:00<?, ?it/s]

usable training spectra: 2539608 | unique molecules: 277566


scaffolds:   0%|          | 0/24 [00:00<?, ?it/s]

train: 2284005 | val: 255603


spectra:   0%|          | 0/24 [00:00<?, ?it/s]

spectra:   0%|          | 0/24 [00:00<?, ?it/s]

spectra:   0%|          | 0/5 [00:00<?, ?it/s]

[after tensors] RAM used 7.57 GB | free 24.86 GB | VRAM 0.00 GB


In [10]:
class MzEmbed(nn.Module):
    """Fixed multi-scale sin/cos features of a mass value, then a learned projection."""
    def __init__(self, d, n_freq=64, lo=1e-3, hi=10.0):
        super().__init__()
        self.register_buffer("freq", torch.logspace(math.log10(lo), math.log10(hi), n_freq) * 2 * math.pi)
        self.proj = nn.Sequential(nn.Linear(2 * n_freq, d), nn.GELU(), nn.Linear(d, d))
    def forward(self, x):
        a = x.unsqueeze(-1) * self.freq
        return self.proj(torch.cat([a.sin(), a.cos()], -1))

class SpecModel(nn.Module):
    def __init__(self, c=CFG):
        super().__init__()
        d = c.D_MODEL
        self.mz_emb, self.loss_emb, self.prec_emb = MzEmbed(d), MzEmbed(d), MzEmbed(d)
        self.int_proj = nn.Sequential(nn.Linear(1, d), nn.GELU(), nn.Linear(d, d))
        self.adduct_emb = nn.Embedding(len(ADDUCT_LIST), d)
        self.ce_proj = nn.Linear(1, d)
        self.cls = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, c.N_HEADS, d * c.FF_MULT, c.DROPOUT,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, c.N_LAYERS, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.fp_head = nn.Sequential(nn.Linear(d, 1024), nn.GELU(), nn.Dropout(c.DROPOUT), nn.Linear(1024, c.NBITS))
        self.spec_proj = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, c.EMB_DIM))
        self.mol_enc = nn.Sequential(nn.Linear(c.NBITS, 1024), nn.GELU(), nn.Linear(1024, c.EMB_DIM))
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))

    def encode_spec(self, mz, inten, pad, prec, adduct, ce):
        tok = self.mz_emb(mz) + self.loss_emb((prec.unsqueeze(1) - mz).clamp(min=0)) \
              + self.int_proj(inten.sqrt().unsqueeze(-1))
        ctx = self.prec_emb(prec) + self.adduct_emb(adduct) + self.ce_proj((ce / 100.0).unsqueeze(-1))
        x = torch.cat([self.cls.expand(len(mz), -1, -1) + ctx.unsqueeze(1), tok], 1)
        key_pad = torch.cat([torch.zeros(len(mz), 1, dtype=torch.bool, device=mz.device), pad], 1)
        h = self.norm(self.enc(x, src_key_padding_mask=key_pad)[:, 0])
        return h

    def forward(self, mz, inten, pad, prec, adduct, ce):
        h = self.encode_spec(mz, inten, pad, prec, adduct, ce)
        return self.fp_head(h), F.normalize(self.spec_proj(h), dim=-1)

    def embed_mol(self, fp):
        return F.normalize(self.mol_enc(fp), dim=-1)

model = SpecModel().to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

parameters: 9.8M


In [11]:
def _flat_mass_chunk(chunk):
    out = []
    for s in chunk:
        m = Chem.MolFromSmiles(s)
        out.append(None if m is None else (Chem.MolToSmiles(m, isomericSmiles=False), Descriptors.ExactMolWt(m)))
    return out

class Library:
    """Mass-sorted candidate library. Fingerprints are bit-packed and can live on disk (mmap_path)."""
    def __init__(self, smiles, desc="library", mmap_path=None, chunk=20000):
        smiles = list(dict.fromkeys(smiles))
        tasks = [smiles[i:i + chunk] for i in range(0, len(smiles), chunk)]
        seen, smi, mass = set(), [], []
        for res in parallel_imap(_flat_mass_chunk, tasks, desc + ": parse"):
            for r in res:
                if r is None or r[0] in seen: continue
                seen.add(r[0]); smi.append(r[0]); mass.append(r[1])
        del seen, smiles, tasks
        order = np.argsort(np.asarray(mass))
        self.mass = np.asarray(mass, dtype=np.float64)[order]
        self.smiles = np.array(smi, dtype=object)[order]
        del smi, mass, order
        n, nb = len(self.smiles), CFG.NBITS // 8
        fp = (np.lib.format.open_memmap(mmap_path, mode="w+", dtype=np.uint8, shape=(n, nb))
              if mmap_path else np.zeros((n, nb), np.uint8))
        tasks = [list(self.smiles[i:i + chunk]) for i in range(0, n, chunk)]   # only pointers, no string copies
        pos = 0
        for packed, _ in parallel_imap(_fp_chunk, tasks, desc + ": fingerprints"):
            fp[pos:pos + len(packed)] = packed; pos += len(packed)
        if mmap_path:
            fp.flush(); del fp
            fp = np.load(mmap_path, mmap_mode="r")        # read-only, paged in from disk on demand
        self.fp = fp
        free()
    def __len__(self): return len(self.smiles)
    def window(self, neutral_mass, ppm):
        lo, hi = neutral_mass * (1 - ppm * 1e-6), neutral_mass * (1 + ppm * 1e-6)
        return int(np.searchsorted(self.mass, lo, "left")), int(np.searchsorted(self.mass, hi, "right"))

MAX_CAND = 20000     # max candidates scored per tier (closest in mass first). Bounds VRAM on huge libraries.

def zscore(x):
    return (x - x.mean()) / (x.std() + 1e-6) if len(x) > 1 else x * 0

@torch.no_grad()
def encode_all(m, T, bs=512):
    m.eval()
    logits, embs = [], []
    for i in range(0, len(T["mz"]), bs):
        b = {k: v[i:i + bs].to(device, non_blocking=True) for k, v in T.items()}
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            lg, em = m(b["mz"], b["inten"], b["pad"], b["prec"], b["adduct"], b["ce"])
        logits.append(lg.float()); embs.append(em.float())
    return torch.cat(logits), torch.cat(embs)

@torch.no_grad()
def lib_emb_cos(m, fps, spec_emb):
    with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        me = m.embed_mol(fps)
    return me.float() @ spec_emb

@torch.no_grad()
def rank_all(m, lib, df, T, topk=CFG.TOPK, alpha=CFG.ALPHA, desc="ranking", chunk=512):
    """Same ranking as before, but encodes and scores 512 spectra at a time."""
    prec = df["precursor_mz"].to_numpy(dtype=np.float64)
    adds = df["adduct"].to_numpy()
    results, pbar = [], tqdm(total=len(df), desc=desc, leave=False)
    for c0 in range(0, len(df), chunk):
        sl = {k: v[c0:c0 + chunk] for k, v in T.items()}
        logits, embs = encode_all(m, sl)
        for j in range(len(logits)):
            i = c0 + j
            neutral = prec[i] - ADDUCT_DELTA[adds[i]]
            chosen, seen_keys, prev = [], set(), (0, 0)
            for ppm in CFG.PPM_TIERS:
                i0, i1 = lib.window(neutral, ppm)
                idx = (np.concatenate([np.arange(i0, prev[0]), np.arange(prev[1], i1)])
                       if prev[1] > prev[0] else np.arange(i0, i1))
                prev = (i0, i1)
                if len(idx) == 0: continue
                if len(idx) > MAX_CAND:
                    idx = idx[np.argsort(np.abs(lib.mass[idx] - neutral))[:MAX_CAND]]
                packed = torch.from_numpy(np.ascontiguousarray(lib.fp[idx])).to(device)
                fps = unpack_gpu(packed).float()                  # 8x less host-to-GPU traffic
                score = zscore(fps @ logits[j]) + alpha * zscore(lib_emb_cos(m, fps, embs[j]))
                for r in torch.argsort(score, descending=True).tolist():
                    smi = lib.smiles[idx[r]]
                    key = ik14(smi)
                    if key is None or key in seen_keys: continue
                    seen_keys.add(key); chosen.append(smi)
                    if len(chosen) >= topk: break
                if len(chosen) >= topk: break
            results.append(chosen)
        pbar.update(len(logits)); del logits, embs
    pbar.close()
    return results

In [12]:
def make_val_library():
    smis = list(va_df["smiles"]) + list(extra_smiles)
    if CFG.VAL_LIB_INCLUDES_TRAIN:
        smis += list(tr_df["smiles"])
    return Library(smis, "val library")

val_lib = make_val_library()
print("validation library size:", len(val_lib))

def augment(b):
    mz, inten, pad = b["mz"], b["inten"], b["pad"]
    real = ~pad
    drop = (torch.rand_like(inten) < CFG.PEAK_DROPOUT) & real
    keep_one = real.sum(1) - drop.sum(1) <= 0         # never drop everything
    drop[keep_one] = False
    pad = pad | drop
    mz = mz + torch.randn_like(mz) * CFG.MZ_JITTER * real
    inten = (inten * (1 + 0.05 * torch.randn_like(inten))).clamp(min=0) * (~pad)
    return mz, inten, pad

def train_model(model, epochs=CFG.EPOCHS):
    need = sum(v.numel() * v.element_size() for v in tr_T.values()) + packed_u.nbytes
    total = torch.cuda.get_device_properties(0).total_memory if device.type == "cuda" else 0
    store = device if (device.type == "cuda" and need < 0.35 * total) else torch.device("cpu")
    print(f"training data {need/1e9:.2f} GB -> stored on {store}")
    TR = {k: v.to(store) for k, v in tr_T.items()}
    MOLP = torch.from_numpy(packed_u).to(store)       # one packed fingerprint per unique molecule
    TRM = torch.from_numpy(tr_mol).to(store)          # spectrum -> molecule row
    n, bs = len(TRM), CFG.BATCH_SIZE
    steps = n // bs

    decay = [p for _, p in model.named_parameters() if p.ndim > 1]
    nodecay = [p for _, p in model.named_parameters() if p.ndim <= 1]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": CFG.WEIGHT_DECAY},
                             {"params": nodecay, "weight_decay": 0.0}], lr=CFG.LR, betas=(0.9, 0.98))
    total_steps = epochs * steps; warm = max(1, int(CFG.WARMUP_FRAC * total_steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total_steps - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)

    best, best_state, hist = -1.0, None, []
    n_eval = min(CFG.EVAL_MAX_N, len(va_df))
    va_sub = va_df.iloc[:n_eval].reset_index(drop=True)
    va_sub_T = {k: v[:n_eval] for k, v in va_T.items()}

    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); tl = tb = tc = 0.0
        perm = torch.randperm(n, device=store)
        for s in range(steps):
            ix = perm[s * bs:(s + 1) * bs]
            g = lambda k: TR[k][ix].to(device, non_blocking=True)
            prec, adduct, ce = g("prec"), g("adduct"), g("ce")
            mz_a, int_a, pad_a = augment(dict(mz=g("mz"), inten=g("inten"), pad=g("pad")))
            fpf = unpack_gpu(MOLP[TRM[ix]].to(device, non_blocking=True)).float()
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                logit, se = model(mz_a, int_a, pad_a, prec, adduct, ce)
                me = model.embed_mol(fpf)
            bce = F.binary_cross_entropy_with_logits(logit.float(), fpf)
            sim = model.logit_scale.exp().clamp(max=100) * se.float() @ me.float().T
            tgt = torch.arange(len(sim), device=device)
            nce = 0.5 * (F.cross_entropy(sim, tgt) + F.cross_entropy(sim.T, tgt))
            loss = bce * 10 + CFG.CONTRASTIVE_W * nce
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
            scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item(); tb += bce.item(); tc += nce.item()
        msg = f"ep {ep:3d}/{epochs} | loss {tl/steps:.4f} | bce {tb/steps:.5f} | nce {tc/steps:.3f} | {time.time()-t0:.1f}s"
        if ep % CFG.EVAL_EVERY == 0 or ep == epochs:
            preds = rank_all(model, val_lib, va_sub, va_sub_T, desc="val")
            r = mrr_at_k(preds, va_sub["smiles"].tolist())
            msg += f" | val MRR@25 {r['MRR@25']:.4f} top1 {r['top1']:.3f} rec@25 {r['recall@25']:.3f}"
            hist.append((ep, r["MRR@25"]))
            if r["MRR@25"] > best:
                best = r["MRR@25"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                torch.save(best_state, "best_model.pt")
        print(msg)
    del TR, MOLP, TRM; free()
    if best_state is not None:
        model.load_state_dict(best_state)
    print("best val MRR@25:", best)
    return model, hist

val library: parse:   0%|          | 0/2 [00:00<?, ?it/s]

val library: fingerprints:   0%|          | 0/2 [00:00<?, ?it/s]

validation library size: 28121


In [13]:
model, hist = train_model(model)

training data 1.42 GB -> stored on cuda
ep   1/7 | loss 1.7866 | bce 0.07745 | nce 2.024 | 2791.7s
ep   2/7 | loss 0.8680 | bce 0.06076 | nce 0.521 | 2798.2s
ep   3/7 | loss 0.7218 | bce 0.05814 | nce 0.281 | 2796.9s
ep   4/7 | loss 0.6509 | bce 0.05628 | nce 0.176 | 2794.7s


val:   0%|          | 0/300 [00:00<?, ?it/s]

ep   5/7 | loss 0.6056 | bce 0.05481 | nce 0.115 | 2796.0s | val MRR@25 0.6590 top1 0.603 rec@25 0.767
ep   6/7 | loss 0.5771 | bce 0.05376 | nce 0.079 | 2794.0s


val:   0%|          | 0/300 [00:00<?, ?it/s]

ep   7/7 | loss 0.5648 | bce 0.05326 | nce 0.064 | 2795.1s | val MRR@25 0.6649 top1 0.613 rec@25 0.767
best val MRR@25: 0.6649267075517076


In [14]:
n_eval = min(500, len(va_df))
va_sub = va_df.iloc[:n_eval].reset_index(drop=True)
va_sub_T = {k: v[:n_eval] for k, v in va_T.items()}
best_alpha, best_score = CFG.ALPHA, -1
for a in [0.0, 0.25, 0.5, 1.0, 2.0]:
    preds = rank_all(model, val_lib, va_sub, va_sub_T, alpha=a, desc=f"alpha={a}")
    r = mrr_at_k(preds, va_sub["smiles"].tolist())
    print(f"alpha {a:4.2f} | MRR@25 {r['MRR@25']:.4f} | top1 {r['top1']:.3f} | top5 {r['top5']:.3f} | recall@25 {r['recall@25']:.3f}")
    if r["MRR@25"] > best_score: best_alpha, best_score = a, r["MRR@25"]
CFG.ALPHA = best_alpha
print("using alpha =", CFG.ALPHA)

alpha=0.0:   0%|          | 0/500 [00:00<?, ?it/s]

alpha 0.00 | MRR@25 0.5934 | top1 0.514 | top5 0.702 | recall@25 0.740


alpha=0.25:   0%|          | 0/500 [00:00<?, ?it/s]

alpha 0.25 | MRR@25 0.6087 | top1 0.536 | top5 0.714 | recall@25 0.740


alpha=0.5:   0%|          | 0/500 [00:00<?, ?it/s]

alpha 0.50 | MRR@25 0.6169 | top1 0.548 | top5 0.718 | recall@25 0.740


alpha=1.0:   0%|          | 0/500 [00:00<?, ?it/s]

alpha 1.00 | MRR@25 0.6164 | top1 0.544 | top5 0.718 | recall@25 0.740


alpha=2.0:   0%|          | 0/500 [00:00<?, ?it/s]

alpha 2.00 | MRR@25 0.6225 | top1 0.556 | top5 0.716 | recall@25 0.740
using alpha = 2.0


In [15]:
final_lib = Library(list(train_df["smiles"]) + list(extra_smiles) + (test_truth or []), "final library")
print("final library size:", len(final_lib))   # note: test_truth is only non-empty in DEMO mode

test_preds = rank_all(model, final_lib, test_df, te_T, alpha=CFG.ALPHA, desc="test")
print("spectra with fewer than 25 candidates:", sum(len(p) < CFG.TOPK for p in test_preds))

final library: parse:   0%|          | 0/14 [00:00<?, ?it/s]

final library: fingerprints:   0%|          | 0/14 [00:00<?, ?it/s]

final library size: 277562


test:   0%|          | 0/1213 [00:00<?, ?it/s]

spectra with fewer than 25 candidates: 0


In [16]:
SAMPLE_SUBMISSION_PATH = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv"  # edit
SUBMISSION_PATH = CFG.SUBMISSION_PATH
N_GUESSES = CFG.TOPK
FALLBACK_SMILES = "CCO"     # every molecule_id must carry at least one guess

def find_id_column(df, sample_ids):
    """Return the test_df column whose values best match the sample submission ids."""
    sample_ids = set(map(str, sample_ids))
    best, best_n = None, 0
    for c in df.columns:
        if c in ("mz", "intensity"):
            continue
        try:
            n = len(sample_ids & set(df[c].astype(str).str.strip()))
        except Exception:
            continue
        if n > best_n:
            best, best_n = c, n
    return best, best_n

def build_submission(test_df, preds, sample_path, k=N_GUESSES):
    sample = pd.read_csv(sample_path)
    sample["molecule_id"] = sample["molecule_id"].astype(str).str.strip()

    id_col, n_match = find_id_column(test_df, sample["molecule_id"])
    print(f"using test_df['{id_col}'] as molecule_id | matches {n_match}/{len(sample)}")
    if id_col is None or n_match < 0.99 * len(sample):
        raise ValueError(
            "test_df has no column that matches the sample submission ids. "
            "Set COLS['id'] to the right column in section 3 (probably 'molecule_id') and rerun the loading cells.")

    ranked = {}
    for mid, p in zip(test_df[id_col].astype(str).str.strip(), preds):
        seen, out = set(), []
        for smi in p:
            if smi and ";" not in smi and smi not in seen:
                seen.add(smi); out.append(smi)
            if len(out) == k:
                break
        ranked[mid] = out

    sub = sample[["molecule_id"]].copy()
    sub["smiles"] = [";".join(ranked.get(m, [])) or FALLBACK_SMILES for m in sub["molecule_id"]]
    return sub

submission = build_submission(test_df, test_preds, SAMPLE_SUBMISSION_PATH)
submission.to_csv(SUBMISSION_PATH, index=False)

n_guesses = submission["smiles"].str.split(";").map(len)
print(f"wrote {len(submission)} rows | guesses per molecule: median {int(n_guesses.median())}, min {n_guesses.min()}, max {n_guesses.max()}")
print("rows on fallback only:", (submission["smiles"] == FALLBACK_SMILES).sum())
submission.head()

using test_df['molecule_id'] as molecule_id | matches 400/400
wrote 400 rows | guesses per molecule: median 25, min 25, max 25
rows on fallback only: 0


,molecule_id,smiles
0,m_005e53,O=C(O)c1cccnc1CN1CCC(c2ccccc2Br)CC1;CC1CC(c2cc...
1,m_006153,COc1ccc(N2CC(C(=O)Nc3nncn3C3CC3)CC2=O)cc1;COc1...
2,m_00b5aa,COc1ccnc(CN2CCOCC3(CCCC3)C2)c1OC;Cc1ccc(O)c(C(...
3,m_01e922,Cc1ccc(C(=O)OCC2CCCCC2)c(=O)[nH]1;Cc1cc(O)ccc1...
4,m_0259d4,CC(C)(C)OC(=O)NC1(c2noc(-c3cccc4[nH]c(=O)[nH]c...
